In [ ]:
import pandas as pd
import numpy as np
RAW_DIR = "../../data/raw"
OUT_DIR = "../../data/python_master"

In [ ]:
with pd.ExcelFile(f"{RAW_DIR}/OBR/efo-march-2026-detailed-forecast-tables-economy.xlsx") as xls:
    hs_raw = pd.read_excel(xls, sheet_name='1.16', skiprows=2, index_col=1,
                       nrows=93)
    rate_raw = pd.read_excel(xls, sheet_name='1.9', skiprows=2, index_col=1,
                         nrows = 93)
    cpi_raw = pd.read_excel(xls, sheet_name='1.7', skiprows=3, index_col=1,
                        nrows=93)

dfs = [hs_raw, rate_raw, cpi_raw]

for df in dfs:
    df.drop(columns="Unnamed: 0", inplace=True)

hs_raw = hs_raw.rename(columns={
    'House price index \n(Jan 2023 = 100)' : "hprice",
    'Residential property transactions \n(000s, seasonally adjusted)' : "vol",
    'Housing stock, UK (000s)' : "stock",
})

hs = hs_raw[["hprice", "vol", "stock"]].copy()

rate_raw = rate_raw.rename(columns={
    "Bank Rate" : "r3"
})

rate = rate_raw["r3"].copy()

cpi = cpi_raw["CPI.1"].copy()

proc = pd.concat([hs, rate, cpi], axis=1)
proc = proc.reset_index()
proc = proc.rename(columns={"CPI.1" : "CPI",
                            "index" : "period"})
proc['period'] = pd.PeriodIndex(proc['period'], freq='Q')
proc 
#OK NOW CONSTRUCTION PROXY AND DEFLATE

In [ ]:
import requests
from bs4 import BeautifulSoup

url = "https://costmodelling.com/construction-indices"

cc_raw = requests.get(url)

cc_parse = BeautifulSoup(cc_raw.text, "html.parser")

# print(cc_parse.prettify())  

table = cc_parse.find("table", class_="indices")
rows = table.find_all("tr")

data = []

for row in rows[1:]:
    cells = row.find_all('td')
    data.append([cells[0].text.strip(), cells[1].text.strip(), cells[2].text.strip()])

df = pd.DataFrame(data, columns=["date", "tpi", "bci"])

df['date'] = df['date'].replace('', pd.NA).ffill()

df['quarter'] = df.groupby('date').cumcount() + 1

df['period'] = pd.PeriodIndex(
    df['date'].astype(str) + 'Q' + df['quarter'].astype(str), 
    freq='Q'
)

cc = df[["period", "bci"]].copy()
cc['bci'] = pd.to_numeric(cc['bci'], errors='coerce')

master = proc.merge(cc, on='period', how='left')

# Forward filling with 2.8% per quarter growth
last_bci = float(master.loc[master['period'] == '2028Q4', 'bci'].values[0])
growth = 1.028 ** (1/4)  # annual 2.8% to quarterly

for i, idx in enumerate(master[master['period'] > '2028Q4'].index, 1):
    master.loc[idx, 'bci'] = last_bci * (growth ** i)

cols = ["period", "lrprc", "lvol", "lstock", "r3", "lrcc"]

master["lrprc"] = np.log(master["hprice"] / master["CPI"] * 100)
master["lvol"] = np.log(master["vol"])
master["lstock"] = np.log(master["stock"])
master["lrcc"] = np.log(master['bci'] / master['CPI'] * 100)

master = master[cols].copy()

master.to_csv(f"{OUT_DIR}/OBR/obr_scenario.csv")


In [ ]:
master